# PDF IRN stamp

Weekly: put source invoices in Drive `source/` and targets in `target/` (filename contains the document number, e.g. `469 box.pdf`), then **Runtime → Run all**.

Results go to `output/` as `{DocNo}_{YYYYMMDDTHHMMSS}_{target_name}` (India time). Already-stamped pairs are skipped. At the end, a ZIP downloads unless you uncheck that in Folders. **Only newly processed download** (on by default) packs PDFs stamped in this run only, not ones skipped as already existing; uncheck it to zip the whole `output/` folder.


In [ ]:
#@title 1. Install packages
%%capture
%pip install PyMuPDF pyzbar pillow qrcode tqdm
# Carlito is metric-compatible with the Calibri used by the invoice template.
!apt-get update && apt-get install -y zbar-tools fonts-crosextra-carlito


In [ ]:
#@title 2. Imports
import io
import os
import re
import shutil
import zipfile
import base64
import json
from datetime import datetime, timedelta, timezone

import fitz  # PyMuPDF
import qrcode
from PIL import Image as PILImage
from pyzbar.pyzbar import decode
from tqdm.auto import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Folders
#@markdown Drive folders. Target filenames must contain the document number (example: `469 box.pdf`).
SOURCE_DIR = "/content/drive/MyDrive/pdf-edit-tool/source/"  #@param {type:"string"}
TARGET_DIR = "/content/drive/MyDrive/pdf-edit-tool/target/"  #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/pdf-edit-tool/output/"  #@param {type:"string"}
#@markdown After the batch, zip output PDFs and download in the browser.
DOWNLOAD_OUTPUT_ZIP = True  #@param {type:"boolean"}
#@markdown If checked, the ZIP has only PDFs stamped in this run (excludes already-existing skips). Uncheck to zip the whole folder.
ONLY_NEWLY_PROCESSED_DOWNLOAD = True  #@param {type:"boolean"}


In [ ]:
#@title 4. Helpers (extraction, layout, matching)
IRN_BOX = {
    "label_x0": 275.46,
    "label_x1": 323.65,
    "value_x1": 462.55,
    "row_tops": (31.68, 65.70, 79.87, 94.04),
    "label_text_x": 277.79,
    "value_text_x": 325.97,
    "label_baselines": (43.14, 77.16, 91.33),
    "irn_first_baseline": 42.26,
    "irn_line_height": 10.50,
    "label_size": 10.0,
    "value_size": 10.0,
    "irn_size": 9.0,
    "line_width": 0.75,
    "text_pad": 1.5,
    "qr_rect": (471.24, 31.32, 559.44, 118.80),
}

CALIBRI_FONT_CANDIDATES = [
    ("carlito",
     "/usr/share/fonts/truetype/crosextra/Carlito-Regular.ttf",
     "/usr/share/fonts/truetype/crosextra/Carlito-Bold.ttf"),
    ("calibri", r"C:\Windows\Fonts\calibri.ttf", r"C:\Windows\Fonts\calibrib.ttf"),
]

IST = timezone(timedelta(hours=5, minutes=30))


def process_pdf_for_qr_and_data(source_pdf_path):
    """Extract QR JWT payload and rebuild a QR image from a source PDF."""
    qr_codes_found = []
    output_dir = "temp_qr_extraction_images"

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    try:
        doc = fitz.open(source_pdf_path)
    except fitz.utils.PyMuPDFError as e:
        print(f"Error opening PDF: {e}. Please ensure the file exists at '{source_pdf_path}'")
        return None, None, None

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(dpi=300)
        img_bytes = pix.pil_tobytes(format="PNG")
        image_pil = PILImage.open(io.BytesIO(img_bytes))
        image_filename = os.path.join(output_dir, f"page_{page_num + 1}.png")
        image_pil.save(image_filename)

    doc.close()

    for filename in os.listdir(output_dir):
        if filename.endswith((".png", ".jpg", ".jpeg")):
            image = PILImage.open(os.path.join(output_dir, filename))
            for obj in decode(image):
                if obj.type == "QRCODE":
                    qr_codes_found.append({"image_file": filename, "data": obj.data.decode("utf-8")})

    shutil.rmtree(output_dir)

    if not qr_codes_found:
        print(f"No QR codes found in '{source_pdf_path}'.")
        return None, None, None

    original_qr_code_content = qr_codes_found[0]["data"]
    parsed_e_invoice_data = None
    full_jwt_payload_dict = None

    try:
        jwt_parts = original_qr_code_content.split(".")
        if len(jwt_parts) < 2:
            raise ValueError("Invalid JWT format: missing payload.")

        payload_encoded = jwt_parts[1]
        padding_needed = 4 - len(payload_encoded) % 4
        if padding_needed != 4:
            payload_encoded += "=" * padding_needed

        payload_decoded = base64.b64decode(payload_encoded).decode("utf-8")
        full_jwt_payload_dict = json.loads(payload_decoded)
        extracted_info_json_str = full_jwt_payload_dict.get("data")
        if extracted_info_json_str:
            parsed_e_invoice_data = json.loads(extracted_info_json_str)
        else:
            print("Could not find 'data' field in the JWT payload.")
    except Exception as e:
        print(f"Error processing JWT data: {e}")

    reconstructed_qr_path = os.path.join(os.getcwd(), "reconstructed_original_qr_code_func.png")
    qrcode.make(original_qr_code_content).save(reconstructed_qr_path)
    return reconstructed_qr_path, parsed_e_invoice_data, full_jwt_payload_dict


def register_box_fonts(page):
    for tag, regular, bold in CALIBRI_FONT_CANDIDATES:
        if os.path.exists(regular) and os.path.exists(bold):
            page.insert_font(fontname=tag, fontfile=regular)
            page.insert_font(fontname=tag + "-bd", fontfile=bold)
            return tag, tag + "-bd", fitz.Font(fontfile=regular)
    return "helv", "hebo", fitz.Font(fontname="helv")


def extract_ack_no(source_pdf_path):
    """Read ACK No. from the invoice text layer (not present in the QR payload)."""
    label_re = re.compile(r"ack\.?\s*no", re.IGNORECASE)
    doc = fitz.open(source_pdf_path)
    try:
        for page in doc:
            spans = [
                span
                for block in page.get_text("dict")["blocks"] if block["type"] == 0
                for line in block["lines"]
                for span in line["spans"]
            ]
            for label in spans:
                if not label_re.search(label["text"]):
                    continue
                label_mid = (label["bbox"][1] + label["bbox"][3]) / 2
                same_row = [
                    s for s in spans
                    if s["bbox"][0] >= label["bbox"][2] - 3
                    and abs((s["bbox"][1] + s["bbox"][3]) / 2 - label_mid) <= 3
                ]
                for span in sorted(same_row, key=lambda s: s["bbox"][0]):
                    value = span["text"].strip().lstrip(":").strip()
                    if re.fullmatch(r"[0-9]{10,20}", value):
                        return value
    finally:
        doc.close()
    return ""


def trim_qr_quiet_zone(qr_image_path):
    img = PILImage.open(qr_image_path).convert("L")
    bbox = img.point(lambda v: 255 if v < 128 else 0).getbbox()
    if bbox is None:
        return qr_image_path
    trimmed = os.path.join(os.path.dirname(os.path.abspath(qr_image_path)), "qr_trimmed.png")
    img.crop(bbox).save(trimmed)
    return trimmed


def wrap_to_width(text, font, size, max_width):
    lines, current = [], ""
    for char in text:
        if font.text_length(current + char, size) <= max_width:
            current += char
        else:
            lines.append(current)
            current = char
    if current:
        lines.append(current)
    return lines


def add_irn_box_to_pdf(target_pdf_path, output_pdf_path, qr_image_path,
                       irn, ack_no, ack_dt, page_num=0):
    box = IRN_BOX
    doc = fitz.open(target_pdf_path)
    page = doc[page_num]
    regular, bold, metrics = register_box_fonts(page)
    black = (0, 0, 0)

    y_top, y_bottom = box["row_tops"][0], box["row_tops"][-1]
    for x in (box["label_x0"], box["label_x1"], box["value_x1"]):
        page.draw_line(fitz.Point(x, y_top), fitz.Point(x, y_bottom),
                       color=black, width=box["line_width"])
    for y in box["row_tops"]:
        page.draw_line(fitz.Point(box["label_x0"], y), fitz.Point(box["value_x1"], y),
                       color=black, width=box["line_width"])

    labels = (" IRN : ", " Ack. No. : ", " Ack. Dt. :  ")
    for label, baseline in zip(labels, box["label_baselines"]):
        page.insert_text(fitz.Point(box["label_text_x"], baseline), label,
                         fontname=bold, fontsize=box["label_size"], color=black)

    max_width = box["value_x1"] - box["value_text_x"] - box["text_pad"]
    for i, line in enumerate(wrap_to_width(irn, metrics, box["irn_size"], max_width)):
        baseline = box["irn_first_baseline"] + i * box["irn_line_height"]
        page.insert_text(fitz.Point(box["value_text_x"], baseline), line,
                         fontname=regular, fontsize=box["irn_size"], color=black)

    for value, baseline in zip((ack_no, ack_dt), box["label_baselines"][1:]):
        page.insert_text(fitz.Point(box["value_text_x"], baseline), str(value),
                         fontname=regular, fontsize=box["value_size"], color=black)

    page.insert_image(fitz.Rect(*box["qr_rect"]), filename=trim_qr_quiet_zone(qr_image_path))
    doc.save(output_pdf_path)
    doc.close()
    tqdm.write(f"Modified PDF saved to: {output_pdf_path}")


def list_pdfs(folder):
    if not os.path.isdir(folder):
        print(f"Folder does not exist: {folder}")
        return []
    return sorted(
        os.path.join(folder, name)
        for name in os.listdir(folder)
        if name.lower().endswith(".pdf") and not name.startswith("~$")
    )


def safe_doc_no(doc_no):
    return re.sub(r'[\\/:*?"<>|]+', "-", str(doc_no).strip())


def doc_no_keys(doc_no):
    raw = str(doc_no).strip()
    keys = [raw] if raw else []
    digits = re.findall(r"\d+", raw)
    if digits and digits[-1] not in keys:
        keys.append(digits[-1])
    return keys


def filename_has_doc_no(filename, doc_no):
    stem = os.path.splitext(os.path.basename(filename))[0]
    for key in doc_no_keys(doc_no):
        if re.search(rf"(?<!\d){re.escape(key)}(?!\d)", stem, flags=re.IGNORECASE):
            return True
    return False


def find_targets_for_doc_no(doc_no, target_paths):
    return [path for path in target_paths if filename_has_doc_no(path, doc_no)]


def existing_output(doc_no, target_filename, output_dir):
    if not os.path.isdir(output_dir):
        return None
    prefix = safe_doc_no(doc_no) + "_"
    suffix = "_" + target_filename
    if not suffix.lower().endswith(".pdf"):
        suffix += ".pdf"
    for name in os.listdir(output_dir):
        if name.startswith(prefix) and name.endswith(suffix):
            return name
    return None


def output_filename(doc_no, target_filename, when=None):
    when = when or datetime.now(IST)
    stamp = when.strftime("%Y%m%dT%H%M%S")
    name = target_filename if target_filename.lower().endswith(".pdf") else target_filename + ".pdf"
    return f"{safe_doc_no(doc_no)}_{stamp}_{name}"


def run_batch():
    global THIS_RUN_OUTPUT_PDFS
    THIS_RUN_OUTPUT_PDFS = []
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    source_pdfs = list_pdfs(SOURCE_DIR)
    target_pdfs = list_pdfs(TARGET_DIR)

    print(f"Source PDFs: {len(source_pdfs)}  |  Target PDFs: {len(target_pdfs)}")
    print(f"Output dir : {OUTPUT_DIR}")

    for source_path in tqdm(source_pdfs, desc="Source PDFs", unit="pdf"):
        source_name = os.path.basename(source_path)
        tqdm.write("=" * 72)
        tqdm.write(f"Source: {source_name}")
        qr_path, e_invoice_data, _jwt = process_pdf_for_qr_and_data(source_path)
        if not e_invoice_data:
            tqdm.write("  SKIP: could not extract QR / e-invoice data.")
            continue

        doc_no = str(e_invoice_data.get("DocNo", "")).strip()
        if not doc_no:
            tqdm.write("  SKIP: QR payload has no DocNo.")
            continue

        matches = find_targets_for_doc_no(doc_no, target_pdfs)
        if not matches:
            tqdm.write(f"  SKIP: no target filename contains document number {doc_no!r}.")
            continue
        if len(matches) > 1:
            names = [os.path.basename(p) for p in matches]
            tqdm.write(f"  SKIP: document number {doc_no!r} matched multiple targets: {names}")
            continue

        target_path = matches[0]
        target_name = os.path.basename(target_path)
        already = existing_output(doc_no, target_name, OUTPUT_DIR)
        if already:
            tqdm.write(f"  SKIP: already processed -> {already}")
            continue

        ack_no = extract_ack_no(source_path)
        out_path = os.path.join(OUTPUT_DIR, output_filename(doc_no, target_name))
        tqdm.write(f"  DocNo {doc_no}  ->  {target_name}")
        tqdm.write(f"  IRN {e_invoice_data.get('Irn', '')}")
        tqdm.write(f"  Ack. No. {ack_no or '(missing)'}  |  Ack. Dt. {e_invoice_data.get('IrnDt', '')}")
        add_irn_box_to_pdf(
            target_path,
            out_path,
            qr_path,
            irn=e_invoice_data["Irn"],
            ack_no=ack_no,
            ack_dt=e_invoice_data.get("IrnDt", ""),
        )
        THIS_RUN_OUTPUT_PDFS.append(out_path)


def download_output_zip():
    if not DOWNLOAD_OUTPUT_ZIP:
        print("ZIP download skipped (DOWNLOAD_OUTPUT_ZIP is False).")
        return
    if not os.path.isdir(OUTPUT_DIR):
        print(f"Nothing to zip; folder does not exist: {OUTPUT_DIR}")
        return

    if ONLY_NEWLY_PROCESSED_DOWNLOAD:
        paths = [
            p for p in dict.fromkeys(THIS_RUN_OUTPUT_PDFS)
            if os.path.isfile(p)
        ]
        if not paths:
            print("No newly stamped PDFs this run; nothing to zip.")
            return
    else:
        paths = [
            os.path.join(OUTPUT_DIR, name)
            for name in sorted(os.listdir(OUTPUT_DIR))
            if name.lower().endswith(".pdf")
        ]
        if not paths:
            print("Output folder is empty; nothing to download.")
            return

    n = len(paths)
    stamp = datetime.now(IST).strftime("%Y%m%dT%H%M%S")
    zip_name = f"{n}docs_{stamp}_output.zip"
    zip_path = os.path.join("/tmp", zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in paths:
            zf.write(path, arcname=os.path.basename(path))
    print(f"Zipped {n} PDF(s) -> {zip_name}")
    from google.colab import files
    files.download(zip_path)


THIS_RUN_OUTPUT_PDFS = []


### 5. Run batch

Match each source **DocNo** to a target filename, stamp IRN / Ack. No. / Ack. Dt. + QR, write to `output/`.


In [ ]:
run_batch()


In [ ]:
#@title 6. Download ZIP
download_output_zip()
